# Part 1: Tabular ensemble
## Load models

In [1]:
import numpy
print(numpy.__version__)
if numpy.__version__ != '2.2.6':
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

2.2.6


In [2]:
!cp -r /kaggle/input/autogluon-package/* /kaggle/working/
!pip install -f --quiet --no-index --find-links='/kaggle/input/autogluon-package' 'autogluon.tabular-1.4.0-py3-none-any.whl'

Looking in links: --quiet, /kaggle/input/autogluon-package
Processing ./autogluon.tabular-1.4.0-py3-none-any.whl
Processing /kaggle/input/autogluon-package/pandas-2.3.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from autogluon.tabular==1.4.0)
Processing /kaggle/input/autogluon-package/scikit_learn-1.7.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (from autogluon.tabular==1.4.0)
Processing /kaggle/input/autogluon-package/autogluon.core-1.4.0-py3-none-any.whl (from autogluon.tabular==1.4.0)
Processing /kaggle/input/autogluon-package/autogluon.features-1.4.0-py3-none-any.whl (from autogluon.tabular==1.4.0)
Processing /kaggle/input/autogluon-package/autogluon.common-1.4.0-py3-none-any.whl (from autogluon.core==1.4.0->autogluon.tabular==1.4.0)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
  Attempting uninstall: pandas
    Found ex

In [3]:
from autogluon.tabular import TabularPredictor

In [4]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
  Attempting uninstall: rdkit
    Found existing installation: rdkit 2025.3.6
    Uninstalling rdkit-2025.3.6:
      Successfully uninstalled rdkit-2025.3.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unimol-tools 0.1.4.post1 requires numpy<2.0.0,>=1.22.4, but you have numpy 2.2.6 which is incompatible.
unimol-tools 0.1.4.post1 requires pandas<2.0.0, but you have pandas 2.3.2 which is incompatible.


In [5]:
import pickle
import json
import glob
import os

MODEL_DIRECTORIES = [
    # '/kaggle/input/tabular-polymer-models/TabularPredictor_20250911_005137/models/TabularPredictor_20250911_005137',
    # '/kaggle/input/tabular-polymer-models/TabularPredictor_20250911_082136/TabularPredictor_20250911_082136',
    '/kaggle/input/tabular-polymer-models/TabularPredictor_20250914_094813/models/TabularPredictor_20250914_094813',
]
TARGET_NAMES = ["Tg", "FFV", "Tc", "Density", "Rg"]
TARGET_NAMES_TO_GROUP_WEIGHTS = {
    # "Tg": [1, 1],
    # "FFV": [1, 1],
    # "Tc": [1, 1],
    # "Density": [1, 1],
    # "Rg": [1, 1],
    "Tg": [1],
    "FFV": [1],
    "Tc": [1],
    "Density": [1],
    "Rg": [1],
}

targets_to_preprocessing_configs: dict[str,list[dict]] = {}
targets_to_selected_features: dict[str,list[dict]] = {}
targets_to_model_groups: dict[list[list]] = {}
targets_to_imputers: dict[list[list]] = {}
for target_name in TARGET_NAMES:
    # LOAD TARGET MODELS & CONFIGS.
    targets_to_preprocessing_configs[target_name] = targets_to_preprocessing_configs.get(target_name, [])
    targets_to_selected_features[target_name] = targets_to_selected_features.get(target_name, [])
    targets_to_model_groups[target_name] = targets_to_model_groups.get(target_name, [])
    targets_to_imputers[target_name] = targets_to_imputers.get(target_name, [])
    for model_directory_path in MODEL_DIRECTORIES:
        # LOAD CONFIG.
        with open(f'{model_directory_path}/{target_name}_features_config.json', 'r') as config_file:
            config = json.load(config_file)
        targets_to_preprocessing_configs[target_name].append(config)

        # LOAD SELECTED FEATURES.
        feature_names_filepaths = glob.glob(f'{model_directory_path}/{target_name}*_features.json')
        if len(feature_names_filepaths) > 0:
            feature_names_filepath = feature_names_filepaths[0]
            with open(feature_names_filepath, 'r') as feature_names_file:
                feature_names = json.load(feature_names_file)
            targets_to_selected_features[target_name].append(feature_names)
        else:
            targets_to_selected_features[target_name].append(None)
        
        # LOAD MODELS.
        model_group = []
        imputers = []
        for file_index, model_path in enumerate(sorted(glob.glob(f'{model_directory_path}/{target_name}*.pkl'))):
            if model_path.endswith('_imputer.pkl'):
                with open(model_path, 'rb') as imputer_file:
                    imputer = pickle.load(imputer_file)[file_index//2] # 2 files per fold
                imputers.append(imputer)
                continue
            
            try:
                with open(model_path, 'rb') as model_file:
                    model = pickle.load(model_file)
                    model_group.append(model)
            except:
                model = TabularPredictor.load(model_path, require_py_version_match=False, require_version_match=False)
                model_group.append(model)
                
        targets_to_model_groups[target_name].append(model_group)


In [6]:
targets_to_model_groups

{'Tg': [[<autogluon.tabular.predictor.predictor.TabularPredictor at 0x7b271fd03c50>]],
 'FFV': [[<autogluon.tabular.predictor.predictor.TabularPredictor at 0x7b271fb63250>]],
 'Tc': [[<autogluon.tabular.predictor.predictor.TabularPredictor at 0x7b271fb62290>]],
 'Density': [[<autogluon.tabular.predictor.predictor.TabularPredictor at 0x7b271fb41e90>]],
 'Rg': [[<autogluon.tabular.predictor.predictor.TabularPredictor at 0x7b271f3c2ad0>]]}

## Data loading + preprocessing helper

In [7]:
import pandas as pd

test_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')
test_df.head()

,id,SMILES
0,1109053969,*Oc1ccc(C=NN=Cc2ccc(Oc3ccc(C(c4ccc(*)cc4)(C(F)...
1,1422188626,*Oc1ccc(C(C)(C)c2ccc(Oc3ccc(C(=O)c4cccc(C(=O)c...
2,2032016830,*c1cccc(OCCCCCCCCOc2cccc(N3C(=O)c4ccc(-c5cccc6...


In [8]:
import math
import multiprocessing
from collections import Counter
from typing import Dict, List, Sequence, Tuple

import networkx as nx
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, rdchem
from tqdm.auto import tqdm
from rdkit.Chem import rdMolDescriptors as rdmd
from rdkit.Chem import AllChem
from functools import lru_cache

from rdkit import RDLogger
RDLogger.logger().setLevel(RDLogger.CRITICAL)

# -----------------------------------------------------------------------------
# Utility helpers
# -----------------------------------------------------------------------------
PROPERTY_NAMES: Tuple[str, ...] = tuple(
    rdMolDescriptors.Properties.GetAvailableProperties()
)
PROPERTY_CALCULATOR = rdMolDescriptors.Properties(PROPERTY_NAMES)

ALL_SIDECHAIN_BACKBONE_FEATURE_NAMES = ['backbone_exactmw', 'backbone_amw', 'backbone_lipinskiHBA', 'backbone_lipinskiHBD', 'backbone_NumRotatableBonds', 'backbone_NumHBD', 'backbone_NumHBA', 'backbone_NumHeavyAtoms', 'backbone_NumAtoms', 'backbone_NumHeteroatoms', 'backbone_NumAmideBonds', 'backbone_FractionCSP3', 'backbone_NumRings', 'backbone_NumAromaticRings', 'backbone_NumAliphaticRings', 'backbone_NumSaturatedRings', 'backbone_NumHeterocycles', 'backbone_NumAromaticHeterocycles', 'backbone_NumSaturatedHeterocycles', 'backbone_NumAliphaticHeterocycles', 'backbone_NumSpiroAtoms', 'backbone_NumBridgeheadAtoms', 'backbone_NumAtomStereoCenters', 'backbone_NumUnspecifiedAtomStereoCenters', 'backbone_labuteASA', 'backbone_tpsa', 'backbone_CrippenClogP', 'backbone_CrippenMR', 'backbone_chi0v', 'backbone_chi1v', 'backbone_chi2v', 'backbone_chi3v', 'backbone_chi4v', 'backbone_chi0n', 'backbone_chi1n', 'backbone_chi2n', 'backbone_chi3n', 'backbone_chi4n', 'backbone_hallKierAlpha', 'backbone_kappa1', 'backbone_kappa2', 'backbone_kappa3', 'backbone_Phi', 'sidechain_exactmw', 'sidechain_amw', 'sidechain_lipinskiHBA', 'sidechain_lipinskiHBD', 'sidechain_NumRotatableBonds', 'sidechain_NumHBD', 'sidechain_NumHBA', 'sidechain_NumHeavyAtoms', 'sidechain_NumAtoms', 'sidechain_NumHeteroatoms', 'sidechain_NumAmideBonds', 'sidechain_FractionCSP3', 'sidechain_NumRings', 'sidechain_NumAromaticRings', 'sidechain_NumAliphaticRings', 'sidechain_NumSaturatedRings', 'sidechain_NumHeterocycles', 'sidechain_NumAromaticHeterocycles', 'sidechain_NumSaturatedHeterocycles', 'sidechain_NumAliphaticHeterocycles', 'sidechain_NumSpiroAtoms', 'sidechain_NumBridgeheadAtoms', 'sidechain_NumAtomStereoCenters', 'sidechain_NumUnspecifiedAtomStereoCenters', 'sidechain_labuteASA', 'sidechain_tpsa', 'sidechain_CrippenClogP', 'sidechain_CrippenMR', 'sidechain_chi0v', 'sidechain_chi1v', 'sidechain_chi2v', 'sidechain_chi3v', 'sidechain_chi4v', 'sidechain_chi0n', 'sidechain_chi1n', 'sidechain_chi2n', 'sidechain_chi3n', 'sidechain_chi4n', 'sidechain_hallKierAlpha', 'sidechain_kappa1', 'sidechain_kappa2', 'sidechain_kappa3', 'sidechain_Phi', 'backbone_aromatic_fraction', 'backbone_aromatic_ring_count', 'backbone_rotatable_density', 'sidechain_rotatable_density', 'relative_rigidity', 'sidechain_mass', 'longest_sidechain_length', 'sidechain_count', 'grafting_density', 'sidechain_spacing_std', 'monomer_vdw_surface', 'backbone_vdw_surface', 'sidechain_vdw_surface', 'backbone_polarizability', 'sidechain_polarizability', 'monomer_polarizability']

IMPORTANT_SIDECHAIN_BACKBONE_FEATURE_NAMES = [
    'grafting_density',
    'relative_rigidity',
    'sidechain_rotatable_density',
    'sidechain_chi1n',
    'sidechain_CrippenClogP',
    'sidechain_chi0n',
    'backbone_aromatic_fraction',
    'backbone_CrippenClogP',
    'sidechain_FractionCSP3',
    'sidechain_Phi',
    'sidechain_kappa3',
    'backbone_FractionCSP3',
    'sidechain_kappa2',
    'longest_sidechain_length',
    'sidechain_chi1v',
    'sidechain_NumAtoms',
    'sidechain_chi2v',
    'sidechain_kappa1',
    'backbone_rotatable_density',
    'sidechain_chi4v'
]

EXTRA_SIDECHAIN_BACKBONE_FEATURE_NAMES = [
    'backbone_mass',
    'sidechain_backbone_mass_ratio',
    'sidechain_backbone_heavy_atom_ratio',
    'sidechain_backbone_tpsa_ratio',
    'simplified_grafting_density',
]

def get_sub_molecule(
    parent_molecule: Chem.Mol,
    atom_indices: Sequence[int]
) -> Chem.Mol:
    """
    Create an RDKit Mol containing *only* `atom_indices` plus the bonds
    between them. Guarantees the result is sanitised even when aromatic
    flags become inconsistent (common when slicing out fragments).
    """
    atom_indices_set = set(atom_indices)
    emol = Chem.RWMol()
    index_map: Dict[int, int] = {}

    # copy atoms
    for orig_idx in atom_indices:
        new_idx = emol.AddAtom(parent_molecule.GetAtomWithIdx(orig_idx))
        index_map[orig_idx] = new_idx

    # copy bonds
    for bond in parent_molecule.GetBonds():
        begin, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        if begin in atom_indices_set and end in atom_indices_set:
            emol.AddBond(
                index_map[begin], index_map[end], bond.GetBondType()
            )

    sub_mol = emol.GetMol()

    try:
        # full sanitisation (fast when it succeeds)
        Chem.SanitizeMol(sub_mol)
    except (rdchem.AtomKekulizeException, rdchem.KekulizeException):
        # fall back: skip kekulisation, then rebuild aromaticity
        light_ops = Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE
        Chem.SanitizeMol(sub_mol, sanitizeOps=light_ops)
        Chem.SetAromaticity(sub_mol, Chem.AromaticityModel.AROMATICITY_DEFAULT)

    # optional: add explicit Hs so heavy‑atom counts stay comparable
    sub_mol = Chem.AddHs(sub_mol)
    return sub_mol


def ensure_ring_info(molecule: Chem.Mol) -> None:
    """
    Populate valence/aromatic caches and perceive rings so that
    descriptor calculators won’t assert on an un‑initialised RingInfo.
    Safe to call repeatedly.
    """
    # Updates valence & implicit/explicit H counts
    molecule.UpdatePropertyCache(strict=False)

    # Fast ring perception that fills RingInfo without touching bond orders
    # (If rings are already perceived this is a no‑op.)
    Chem.FastFindRings(molecule)

def compute_rdkit_descriptors(molecule: Chem.Mol) -> Dict[str, float]:
    """
    Compute the full RDKit descriptor vector for `molecule`.
    Guarantees that RingInfo is initialised first.
    """
    ensure_ring_info(molecule)
    values = PROPERTY_CALCULATOR.ComputeProperties(molecule)
    return dict(zip(PROPERTY_NAMES, values))


# -----------------------------------------------------------------------------
# Backbone / side‑chain identification
# -----------------------------------------------------------------------------
def process_polymer_smiles(
    smiles_string: str
) -> Tuple[Chem.Mol | None, List[int]]:
    """
    Strip out the two '[*]' dummy atoms and return:

    • cleaned RDKit Mol
    • indices of the two attachment atoms (after removal)

    If parsing fails, returns (None, []).
    """
    molecule = Chem.MolFromSmiles(smiles_string)
    if molecule is None:
        return None, []

    star_neighbors: List[int] = []
    indices_to_delete: List[int] = []

    for atom in molecule.GetAtoms():
        if atom.GetAtomicNum() == 0:  # star / attachment marker
            star_neighbors.extend(neigh.GetIdx() for neigh in atom.GetNeighbors())
            indices_to_delete.append(atom.GetIdx())

    editable = Chem.RWMol(molecule)
    for idx in sorted(indices_to_delete, reverse=True):
        editable.RemoveAtom(idx)

    cleaned_mol = editable.GetMol()

    # Remap neighbor indices after deletions
    adjusted_neighbors: List[int] = []
    for original_idx in star_neighbors:
        removed_before = sum(1 for deleted in indices_to_delete if deleted < original_idx)
        adjusted_neighbors.append(original_idx - removed_before)

    return cleaned_mol, list(dict.fromkeys(adjusted_neighbors))  # unique & ordered


def identify_backbone_and_sidechains(
    cleaned_molecule: Chem.Mol,
    attachment_indices: List[int],
) -> Tuple[List[int], List[List[int]]]:
    """
    Return indices of backbone atoms and a list of side‑chain index lists.
    Falls back gracefully when the two attachment sites are disconnected.
    """
    num_atoms: int = cleaned_molecule.GetNumAtoms()

    # ----------------------------------------------
    # 1. trivial cases
    # ----------------------------------------------
    if len(attachment_indices) < 2:
        return list(range(num_atoms)), []

    adjacency_matrix = Chem.GetAdjacencyMatrix(cleaned_molecule)
    graph = nx.from_numpy_array(adjacency_matrix)

    # ----------------------------------------------
    # 2. try the normal shortest‑path backbone
    # ----------------------------------------------
    try:
        backbone_path: List[int] = nx.shortest_path(
            graph, attachment_indices[0], attachment_indices[-1]
        )
    except nx.NetworkXNoPath:               # ← add this block
        # Two attachment atoms live in different fragments.
        # Treat the entire molecule as backbone (no side‑chains),
        # but *do* log the situation so you can inspect later if needed.
        # A production system could write to logging.warning instead.
        # print(
        #     f"[WARN] Disconnected attachment points in SMILES → "
        #     f"using whole molecule as backbone."
        # )
        return list(range(num_atoms)), []

    # ----------------------------------------------
    # 3. collect side‑chains as before
    # ----------------------------------------------
    backbone_set = set(backbone_path)
    visited = set(backbone_path)
    sidechain_indices_list: List[List[int]] = []

    for backbone_atom in backbone_path:
        for neighbor in graph.neighbors(backbone_atom):
            if neighbor in backbone_set or neighbor in visited:
                continue
            queue = [neighbor]
            current_chain: List[int] = []
            while queue:
                atom = queue.pop()
                if atom in visited or atom in backbone_set:
                    continue
                visited.add(atom)
                current_chain.append(atom)
                queue.extend(
                    neigh
                    for neigh in graph.neighbors(atom)
                    if neigh not in visited and neigh not in backbone_set
                )
            if current_chain:
                sidechain_indices_list.append(current_chain)

    return backbone_path, sidechain_indices_list


# -----------------------------------------------------------------------------
# Fragment‑specific feature calculations
# -----------------------------------------------------------------------------
def heavy_atom_indices(molecule: Chem.Mol, indices: Sequence[int]) -> List[int]:
    return [
        idx for idx in indices
        if molecule.GetAtomWithIdx(idx).GetAtomicNum() > 1
    ]


def count_aromatic_rings(molecule: Chem.Mol, atom_indices: Sequence[int]) -> int:
    """
    Count *rings* (not atoms) in which at least half the atoms lie on `atom_indices`
    and the ring is aromatic.
    """
    ri = molecule.GetRingInfo()
    rings = ri.AtomRings()
    backbone_set = set(atom_indices)
    ring_count = 0
    for ring in rings:
        if all(molecule.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
            overlap = sum(1 for i in ring if i in backbone_set)
            if overlap >= len(ring) // 2:
                ring_count += 1
    return ring_count


def rotatable_bond_density(
    molecule: Chem.Mol,
    atom_indices: Sequence[int]
) -> float:
    """
    Rotatable bonds per heavy atom within the substructure defined by `atom_indices`.
    """
    sub_mol = get_sub_molecule(molecule, atom_indices)
    rotatable_bonds = AllChem.CalcNumRotatableBonds(sub_mol, strict=True)
    heavy_atoms = sum(
        1 for idx in atom_indices
        if molecule.GetAtomWithIdx(idx).GetAtomicNum() > 1
    )
    return rotatable_bonds / heavy_atoms if heavy_atoms else 0.0


def total_mass(molecule: Chem.Mol, atom_indices: Sequence[int]) -> float:
    """
    Sum of atomic weights (isotope‑aware) for the selected atoms.
    Uses Atom.GetMass() so it works across all RDKit versions.
    """
    return sum(
        molecule.GetAtomWithIdx(idx).GetMass()
        for idx in atom_indices
    )


def sidechain_spacing_std(attachment_indices: List[int]) -> float:
    """
    Standard deviation of attachment points along the backbone.
    The attachment indices are assumed to be in backbone order.
    """
    if len(attachment_indices) < 3:
        return 0.0
    differences = np.diff(sorted(attachment_indices))
    return float(np.std(differences, ddof=1))


def labute_asa(molecule: Chem.Mol) -> float:
    asa = rdmd.CalcLabuteASA(molecule, includeHs=False)
    return asa


def mol_volume(molecule: Chem.Mol) -> float:
    # ---------- fast path (unchanged) ----------
    if hasattr(rdmd, "CalcMolVolume"):
        try:
            ensure_ring_info(molecule)
            return rdmd.CalcMolVolume(molecule)
        except (rdchem.ConformerException, RuntimeError):
            pass   # fall through

    # ---------- embed a conformer --------------
    mol3d = Chem.Mol(molecule)                     # copy
    mol3d = Chem.AddHs(mol3d, addCoords=True)
    ensure_ring_info(mol3d)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    try:
        if AllChem.EmbedMolecule(mol3d, params) != 0:
            return 0.0
    except (rdchem.KekulizeException, rdchem.AtomKekulizeException):
        # ETKDG failed because of broken aromatic flags → skip
        return 0.0

    try:                                           # UFF is optional
        AllChem.UFFOptimizeMolecule(mol3d, maxIters=50)
    except Exception:
        pass

    ensure_ring_info(mol3d)
    try:
        return AllChem.ComputeMolVolume(mol3d)
    except Exception:
        return 0.0


MILLER_POLARIZABILITY: dict[int, float] = {
    1: 0.666,  6: 1.75, 7: 1.10, 8: 0.802, 9: 0.557,
    15: 3.63, 16: 2.90, 17: 2.18, 35: 3.05, 53: 5.35,
}

def miller_polarizability(molecule: Chem.Mol, atom_indices: Sequence[int]) -> float:
    return sum(
        MILLER_POLARIZABILITY.get(molecule.GetAtomWithIdx(idx).GetAtomicNum(), 0.0)
        for idx in atom_indices
    )


# -----------------------------------------------------------------------------
# Master feature extraction
# -----------------------------------------------------------------------------
@lru_cache(50_000)
def extract_sidechain_and_backbone_features(smiles_string: str) -> Dict[str, float]:
    """
    Compute global, backbone, side‑chain and custom cross‑fragment features.
    """
    features: Dict[str, float] = {"SMILES": smiles_string}

    cleaned_mol, attachment_indices = process_polymer_smiles(smiles_string)
    # if cleaned_mol is None:
    #     return features

    backbone_indices, sidechains = identify_backbone_and_sidechains(
        cleaned_mol, attachment_indices
    )
    sidechain_indices_flat = [idx for chain in sidechains for idx in chain]

    # ------------------------------------------------------------------
    # RDKit descriptor blocks
    # ------------------------------------------------------------------
    features.update(
        {
            f"backbone_{name}": value
            for name, value in compute_rdkit_descriptors(
                get_sub_molecule(cleaned_mol, backbone_indices)
            ).items()
        }
    )

    if sidechain_indices_flat:
        sidechain_submol = get_sub_molecule(cleaned_mol, sidechain_indices_flat)
        features.update(
            {
                f"sidechain_{name}": value
                for name, value in compute_rdkit_descriptors(sidechain_submol).items()
            }
        )
    else:
        # Fill zero so downstream code doesn’t run into KeyErrors
        for name in PROPERTY_NAMES:
            features[f"sidechain_{name}"] = 0.0

    # ------------------------------------------------------------------
    # Custom fragment features
    # ------------------------------------------------------------------
    heavy_backbone_atoms = heavy_atom_indices(cleaned_mol, backbone_indices)
    heavy_sidechain_atoms = heavy_atom_indices(cleaned_mol, sidechain_indices_flat)

    # Aromatic metrics
    features["backbone_aromatic_fraction"] = (
        sum(
            1 for idx in heavy_backbone_atoms
            if cleaned_mol.GetAtomWithIdx(idx).GetIsAromatic()
        ) / len(heavy_backbone_atoms) if heavy_backbone_atoms else 0.0
    )
    features["backbone_aromatic_ring_count"] = count_aromatic_rings(
        cleaned_mol, backbone_indices
    )

    # Rotatable bond densities & relative rigidity
    backbone_rot_density = rotatable_bond_density(cleaned_mol, backbone_indices)
    sidechain_rot_density = rotatable_bond_density(cleaned_mol, sidechain_indices_flat)
    features["backbone_rotatable_density"] = backbone_rot_density
    features["sidechain_rotatable_density"] = sidechain_rot_density
    features["relative_rigidity"] = backbone_rot_density - sidechain_rot_density

    # Mass & size descriptors
    features["sidechain_mass"] = total_mass(cleaned_mol, sidechain_indices_flat)
    features["backbone_mass"] = total_mass(cleaned_mol, backbone_indices)
    features["longest_sidechain_length"] = (
        max((len(chain) for chain in sidechains), default=0)
    )

    features["sidechain_count"] = len(sidechains)

    # Grafting metrics
    backbone_heavy_atom_count = len(heavy_backbone_atoms)
    graft_sites = len(sidechains)
    features["grafting_density"] = (
        graft_sites / backbone_heavy_atom_count if backbone_heavy_atom_count else 0.0
    )
    features["sidechain_spacing_std"] = sidechain_spacing_std(attachment_indices)

    # --- van‑der‑Waals surface & volume for each fragment -------------
    features["monomer_vdw_surface"] = labute_asa(cleaned_mol)
    features["backbone_vdw_surface"] = labute_asa(
        get_sub_molecule(cleaned_mol, backbone_indices)
    )
    features["sidechain_vdw_surface"] = (
        labute_asa(get_sub_molecule(cleaned_mol, sidechain_indices_flat))
        if sidechain_indices_flat else 0.0
    )

    # Slow:
    # features["monomer_vdw_volume"] = mol_volume(cleaned_mol)
    # features["backbone_vdw_volume"] = mol_volume(
    #     get_sub_molecule(cleaned_mol, backbone_indices)
    # )
    # features["sidechain_vdw_volume"] = (
    #     mol_volume(get_sub_molecule(cleaned_mol, sidechain_indices_flat))
    #     if sidechain_indices_flat else 0.0
    # )

    # --- Miller polarizability ---------------------------------------
    features["backbone_polarizability"] = miller_polarizability(
        cleaned_mol, backbone_indices
    )
    features["sidechain_polarizability"] = miller_polarizability(
        cleaned_mol, sidechain_indices_flat
    )
    features["monomer_polarizability"] = (
        features["backbone_polarizability"] + features["sidechain_polarizability"]
    )

    # --- Gemini Suggestions -----------------------------------------
    features["sidechain_backbone_mass_ratio"] = features["sidechain_mass"] / (features["backbone_mass"] + 1e6)
    features["sidechain_backbone_heavy_atom_ratio"] = len(heavy_sidechain_atoms) / (len(heavy_backbone_atoms) + 1e6)
    features["sidechain_backbone_tpsa_ratio"] = features["sidechain_tpsa"] / (features["backbone_tpsa"] + 1e6)
    features["simplified_grafting_density"] = features["sidechain_count"] / (len(backbone_indices) + 1e6)

    return features

In [9]:
from typing import Dict
import os

import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdmolops
from rdkit.Chem import rdPartialCharges
from joblib import Memory

ALL_GEMINI_FEATURE_NAMES = ['element_fraction_C', 'element_fraction_N', 'element_fraction_O', 'halogen_count', 'halogen_fraction', 'aromatic_bond_fraction', 'double_bond_fraction', 'triple_bond_fraction', 'wiener_index', 'ring_atom_fraction', 'gasteiger_charge_mean', 'gasteiger_charge_std', 'gasteiger_charge_max_pos', 'gasteiger_charge_max_neg', 'count_ester_carbonate', 'count_sulfone_sulfonamide', 'count_ether_non_aromatic']
IMPORTANT_GEMINI_FEATURE_NAMES = ['element_fraction_C', 
 'element_fraction_O',
 'double_bond_fraction',
 'ring_atom_fraction',
 'element_fraction_N',
 'aromatic_bond_fraction',
 'gasteiger_charge_max_neg',
 'halogen_fraction',
 'wiener_index',
 'gasteiger_charge_mean']

def wiener_index(m):
    res = 0
    amat = Chem.GetDistanceMatrix(m)
    num_atoms = m.GetNumAtoms()
    for i in range(num_atoms):
        for j in range(i+1,num_atoms):
            res += amat[i][j]
    return res

_PRED_CACHE_DIR = os.path.expanduser("~/.cache/gemini_features")
@Memory(location=_PRED_CACHE_DIR, verbose=0).cache
def compute_inexpensive_features(mol: Chem.Mol) -> Dict[str, float]:
    """
    Computes a set of computationally inexpensive features for a given
    RDKit molecule.

    This includes elemental/bond composition, simplified topological descriptors,
    and physicochemical proxies based on partial charges and SMARTS patterns.

    Args:
        mol: An RDKit molecule object.

    Returns:
        A dictionary mapping feature names to their calculated values.
    """
    if mol is None:
        return {}

    features: Dict[str, float] = {}
    num_heavy_atoms = mol.GetNumHeavyAtoms()
    
    # Pre-calculate properties to avoid re-computation
    mol.UpdatePropertyCache(strict=False)
    Chem.FastFindRings(mol)

    # =========================================================================
    # Section 1: Elemental & Bond Composition Features
    # =========================================================================

    # --- Element Fractions ---
    element_counts = {atom.GetAtomicNum(): 0 for atom in mol.GetAtoms()}
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() > 1: # Only count heavy atoms
             element_counts[atom.GetAtomicNum()] = element_counts.get(atom.GetAtomicNum(), 0) + 1

    features['element_fraction_C'] = element_counts.get(6, 0) / num_heavy_atoms if num_heavy_atoms > 0 else 0
    features['element_fraction_N'] = element_counts.get(7, 0) / num_heavy_atoms if num_heavy_atoms > 0 else 0
    features['element_fraction_O'] = element_counts.get(8, 0) / num_heavy_atoms if num_heavy_atoms > 0 else 0

    # --- Halogen Count & Fraction ---
    halogen_atomic_nums = {9, 17, 35, 53}  # F, Cl, Br, I
    halogen_count = sum(count for atomic_num, count in element_counts.items() if atomic_num in halogen_atomic_nums)
    features['halogen_count'] = halogen_count
    features['halogen_fraction'] = halogen_count / num_heavy_atoms if num_heavy_atoms > 0 else 0

    # --- Bond Type Ratios ---
    num_bonds = mol.GetNumBonds()
    if num_bonds > 0:
        num_aromatic_bonds = sum(1 for bond in mol.GetBonds() if bond.GetIsAromatic())
        num_double_bonds = sum(1 for bond in mol.GetBonds() if bond.GetBondType() == Chem.BondType.DOUBLE)
        num_triple_bonds = sum(1 for bond in mol.GetBonds() if bond.GetBondType() == Chem.BondType.TRIPLE)
        
        num_non_aromatic_bonds = num_bonds - num_aromatic_bonds

        features['aromatic_bond_fraction'] = num_aromatic_bonds / num_bonds
        features['double_bond_fraction'] = num_double_bonds / num_non_aromatic_bonds if num_non_aromatic_bonds > 0 else 0
        features['triple_bond_fraction'] = num_triple_bonds / num_non_aromatic_bonds if num_non_aromatic_bonds > 0 else 0
    else:
        features['aromatic_bond_fraction'] = 0
        features['double_bond_fraction'] = 0
        features['triple_bond_fraction'] = 0


    # =========================================================================
    # Section 2: Simplified Topological & Shape Descriptors
    # =========================================================================

    # --- Wiener Index ---
    # The sum of the shortest paths between all pairs of heavy atoms.
    features['wiener_index'] = wiener_index(mol)

    # --- Ring Atom Fraction ---
    # The number of atoms that are part of any ring.
    ring_info = mol.GetRingInfo()
    unique_ring_atoms = set()
    for ring in ring_info.AtomRings():
        unique_ring_atoms.update(ring)
    
    features['ring_atom_fraction'] = len(unique_ring_atoms) / num_heavy_atoms if num_heavy_atoms > 0 else 0


    # =========================================================================
    # Section 3: Fast Physicochemical Proxies
    # =========================================================================

    # --- Gasteiger Partial Charges Statistics ---
    try:
        rdPartialCharges.ComputeGasteigerCharges(mol, nIter=12)
        charges = [atom.GetDoubleProp('_GasteigerCharge') for atom in mol.GetAtoms() if not np.isnan(atom.GetDoubleProp('_GasteigerCharge'))]
        if charges:
            features['gasteiger_charge_mean'] = np.mean(charges)
            features['gasteiger_charge_std'] = np.std(charges)
            features['gasteiger_charge_max_pos'] = max(c for c in charges if c > 0) if any(c > 0 for c in charges) else 0
            features['gasteiger_charge_max_neg'] = min(c for c in charges if c < 0) if any(c < 0 for c in charges) else 0
        else:
             features.update({'gasteiger_charge_mean': 0, 'gasteiger_charge_std': 0, 'gasteiger_charge_max_pos': 0, 'gasteiger_charge_max_neg': 0})
    except Exception: # Handle cases where charge calculation fails
        features.update({'gasteiger_charge_mean': 0, 'gasteiger_charge_std': 0, 'gasteiger_charge_max_pos': 0, 'gasteiger_charge_max_neg': 0})
        
    # --- SMARTS-based Functional Group Counts ---
    smarts_patterns = {
        'count_ester_carbonate': Chem.MolFromSmarts('[CX3](=O)[OX2]'),
        'count_sulfone_sulfonamide': Chem.MolFromSmarts('[SD4](=O)(=O)'),
        'count_ether_non_aromatic': Chem.MolFromSmarts('[OD2]([C;!$(C=O)])[C;!$(C=O)]')
    }

    for name, pattern in smarts_patterns.items():
        if pattern:
            features[name] = len(mol.GetSubstructMatches(pattern))
        else: # Should not happen with valid SMARTS
            features[name] = 0
            
    return features

In [10]:
import polars as pl
from functools import lru_cache
from rdkit import Chem
from rdkit.Chem import Descriptors, GraphDescriptors, MACCSkeys, rdFingerprintGenerator, AllChem, rdmolops
from rdkit.ML.Descriptors import MoleculeDescriptors
import networkx as nx
import joblib
from joblib import Memory
from sentence_transformers import SentenceTransformer
import torch

debug = False

@lru_cache(maxsize=1_000_000)
def get_feature_vector(
        smiles: str,
        morgan_fingerprint_dim: int,
        atom_pair_fingerprint_dim: int,
        torsion_dim: int,
        use_maccs_keys: bool,
        use_graph_features: bool,
        backbone_sidechain_detail_level: int,
        use_extra_backbone_sidechain_features: bool,
        gemini_features_detail_level: int):
    # PARSE SMILES.
    mol = Chem.MolFromSmiles(smiles)
    
    # GET DESCRIPTORS.
    descriptor_names = [descriptor[0] for descriptor in Descriptors._descList]
    descriptor_generator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
    descriptors = np.array(descriptor_generator.CalcDescriptors(mol))

    # print('descriptors:', len(descriptors))

    # GET MORGAN FINGERPRINT.
    morgan_fingerprint = np.array([])
    if morgan_fingerprint_dim > 0:
        morgan_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=morgan_fingerprint_dim)
        morgan_fingerprint = list(morgan_generator.GetFingerprint(mol))

    # print('morgan_fingerprint:', len(morgan_fingerprint))

    # GET ATOM PAIR FINGERPRINT.
    atom_pair_fingerprint = np.array([])
    if atom_pair_fingerprint_dim > 0:
        atom_pair_generator = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=atom_pair_fingerprint_dim)
        atom_pair_fingerprint = list(atom_pair_generator.GetFingerprint(mol))

    # print('atom_pair_fingerprint:', len(atom_pair_fingerprint))

    # GET MACCS.
    maccs_keys = np.array([])
    if use_maccs_keys:
        maccs_keys = MACCSkeys.GenMACCSKeys(mol)
        maccs_keys = list(maccs_keys)

    # print('maccs_keys:', len(maccs_keys))

    # GET TORSION FINGERPRINT.
    torsion_fingerprint = np.array([])
    if torsion_dim > 0:
        torsion_generator = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=torsion_dim)
        torsion_fingerprint = list(torsion_generator.GetFingerprint(mol))

    # print('torsion_fingerprint:', len(torsion_fingerprint))

    # GET GRAPH FEATURES.
    graph_features = []
    if use_graph_features:
        adjacency_matrix = rdmolops.GetAdjacencyMatrix(mol)
        graph = nx.from_numpy_array(adjacency_matrix)
        graph_diameter = nx.diameter(graph) if nx.is_connected(graph) else 0
        avg_shortest_path = nx.average_shortest_path_length(graph) if nx.is_connected(graph) else 0
        cycle_count = len(list(nx.cycle_basis(graph)))
        graph_features = [graph_diameter, avg_shortest_path, cycle_count]

    # print('graph_features:', len(graph_features))

    # GET SIDECHAIN & BACKBONE FEATURES.
    extra_sidechain_backbone_feature_names = EXTRA_SIDECHAIN_BACKBONE_FEATURE_NAMES if use_extra_backbone_sidechain_features else []
    if (backbone_sidechain_detail_level == 0) and (not use_extra_backbone_sidechain_features):
        sidechain_backbone_features = []
    elif (backbone_sidechain_detail_level == 0) and use_extra_backbone_sidechain_features:
        sidechain_backbone_features = extract_sidechain_and_backbone_features(smiles)
        sidechain_backbone_features = [sidechain_backbone_features[name] for name in extra_sidechain_backbone_feature_names]
    elif backbone_sidechain_detail_level == 1:
        sidechain_backbone_features = extract_sidechain_and_backbone_features(smiles)
        sidechain_backbone_features = [sidechain_backbone_features[name] for name in IMPORTANT_SIDECHAIN_BACKBONE_FEATURE_NAMES + extra_sidechain_backbone_feature_names]
    elif backbone_sidechain_detail_level == 2:
        sidechain_backbone_features = extract_sidechain_and_backbone_features(smiles)
        sidechain_backbone_features = [sidechain_backbone_features[name] for name in ALL_SIDECHAIN_BACKBONE_FEATURE_NAMES + extra_sidechain_backbone_feature_names]
    else:
        assert False, f'Invalid backbone vs. sidechain detail level: {backbone_sidechain_detail_level}'

    # print('sidechain_backbone_features:', len(sidechain_backbone_features))

    # GET GEMINI FEATURES.
    if gemini_features_detail_level == 0:
        gemini_features = []
    elif gemini_features_detail_level == 1:
        gemini_features = compute_inexpensive_features(mol)
        gemini_features = [gemini_features[name] for name in IMPORTANT_GEMINI_FEATURE_NAMES]
    elif gemini_features_detail_level == 2:
        gemini_features = compute_inexpensive_features(mol)
        gemini_features = [gemini_features[name] for name in ALL_GEMINI_FEATURE_NAMES]
    else:
        assert False, f'Invalid backbone vs. sidechain detail level: {backbone_sidechain_detail_level}'

    # print('descriptors:', len(descriptors))

    # CONCATENATE FEATURES.
    features = np.concatenate([
        descriptors, 
        morgan_fingerprint, 
        atom_pair_fingerprint, 
        maccs_keys, 
        torsion_fingerprint,
        graph_features,
        sidechain_backbone_features,
        gemini_features
    ])
    return features

def _get_standard_features_dataframe(
        smiles_df: pd.DataFrame, 
        morgan_fingerprint_dim: int,
        atom_pair_fingerprint_dim: int,
        torsion_dim: int,
        use_maccs_keys: bool,
        use_graph_features: bool,
        backbone_sidechain_detail_level: int,
        use_extra_backbone_sidechain_features: bool,
        gemini_features_detail_level: int) -> pd.DataFrame:
    # GET FEATURE NAMES.
    descriptor_names = [descriptor[0] for descriptor in Descriptors._descList]
    morgan_col_names = [f'mfp_{i}' for i in range(morgan_fingerprint_dim)]
    atom_pair_col_names = [f'ap_{i}' for i in range(atom_pair_fingerprint_dim)]
    maccs_col_names = [f'maccs_{i}' for i in range(167)] if use_maccs_keys else []
    torsion_col_names = [f'tt_{i}' for i in range(torsion_dim)]
    graph_col_names = ['graph_diameter', 'avg_shortest_path', 'num_cycles'] if use_graph_features else []
    extra_sidechain_col_names = EXTRA_SIDECHAIN_BACKBONE_FEATURE_NAMES if use_extra_backbone_sidechain_features else []
    sidechain_col_names = [[], IMPORTANT_SIDECHAIN_BACKBONE_FEATURE_NAMES, ALL_SIDECHAIN_BACKBONE_FEATURE_NAMES][backbone_sidechain_detail_level] + extra_sidechain_col_names
    gemini_col_names = [[], IMPORTANT_GEMINI_FEATURE_NAMES, ALL_GEMINI_FEATURE_NAMES][gemini_features_detail_level]
    feature_col_names = descriptor_names + morgan_col_names + atom_pair_col_names + maccs_col_names + torsion_col_names + graph_col_names + sidechain_col_names + gemini_col_names

    # print('descriptor_names:', len(descriptor_names))
    # print('morgan_col_names:', len(morgan_col_names))
    # print('atom_pair_col_names:', len(atom_pair_col_names))
    # print('maccs_col_names:', len(maccs_col_names))
    # print('torsion_col_names:', len(torsion_col_names))
    # print('graph_col_names:', len(graph_col_names))
    # print('extra_sidechain_col_names:', len(extra_sidechain_col_names))
    # print('sidechain_col_names:', len(sidechain_col_names))
    # print('gemini_col_names:', len(gemini_col_names))
    
    # GET FEATURES.
    try:
        features_df = pd.DataFrame(
            np.vstack([
                get_feature_vector(
                    smiles,
                    morgan_fingerprint_dim,
                    atom_pair_fingerprint_dim,
                    torsion_dim,
                    use_maccs_keys,
                    use_graph_features,
                    backbone_sidechain_detail_level,
                    use_extra_backbone_sidechain_features,
                    gemini_features_detail_level
                ) 
                for smiles 
                in smiles_df['SMILES']]),
            columns=feature_col_names
        )
    except:
        global debug
        debug=True
        features_df = pd.DataFrame(
            np.vstack([
                get_feature_vector(
                    smiles,
                    morgan_fingerprint_dim,
                    atom_pair_fingerprint_dim,
                    torsion_dim,
                    use_maccs_keys,
                    use_graph_features,
                    backbone_sidechain_detail_level,
                    use_extra_backbone_sidechain_features,
                    gemini_features_detail_level
                ) 
                for smiles 
                in smiles_df['SMILES']]),
            columns=feature_col_names
        )

    # CLEAN FEATURES.
    f32_max = np.finfo(np.float32).max
    features_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    features_df[features_df > f32_max] = np.nan
    features_df[features_df < -f32_max] = np.nan

    return features_df

_PRED_CACHE_DIR = "predicted_features"
@Memory(location=_PRED_CACHE_DIR, verbose=0).cache
def _get_predicted_features_dataframe(smiles_df: pd.DataFrame, models_directory_path: str) -> pd.DataFrame:
    # LOAD MODELS.
    model_paths = glob.glob(os.path.join(models_directory_path, '*.joblib'))
    model_paths = sorted(model_paths)
    models = [joblib.load(model_path) for model_path in model_paths]

    # LOAD FEATURES CONFIG.
    features_config_path = os.path.join(models_directory_path, 'features_config.json')
    with open(features_config_path, 'r') as features_file:
        features_config = json.load(features_file)

    # COMPUTE INPUT FEATURES.
    input_features_df = _get_standard_features_dataframe(
        smiles_df, 
        **features_config, 
        gemini_features_detail_level=0, 
        use_extra_backbone_sidechain_features=False
    )

    # COMPUTE PREDICTED FEATURES.
    predicted_features_df = pd.DataFrame()
    for model_path, model in zip(model_paths, models):
        predictions = model.predict(input_features_df)
        col_name = os.path.splitext(os.path.basename(model_path))[0]
        predicted_features_df[col_name] = predictions

    return predicted_features_df

def load_backbone_into_sentence_transformer(
        base_model_name_or_path: str,
        finetuned_state_dict_path: str,
        device: str | torch.device = "cpu"
    ) -> SentenceTransformer:
    sentence_transformer_model = SentenceTransformer(base_model_name_or_path, device=str(device))

    finetuned_state_dict = torch.load(finetuned_state_dict_path, map_location="cpu")

    remapped_state_dict = {}
    for parameter_name, tensor in finetuned_state_dict.items():
        # Keep only backbone params; drop custom heads like pooler/output
        if parameter_name.startswith("backbone."):
            stripped_name = parameter_name[len("backbone."):]
            stripped_name = stripped_name.replace('encoder', '0.auto_model.encoder')
            stripped_name = stripped_name.replace('embeddings', '0.auto_model.embeddings')
            stripped_name = stripped_name.replace('word_0.auto_model.embeddings', 'word_embeddings')
            stripped_name = stripped_name.replace('position_0.auto_model.embeddings', 'position_embeddings')
            remapped_state_dict[stripped_name] = tensor
        else:
            # If you happened to save only the HF backbone (no prefix), pass them through
            # and let strict=False ignore anything that doesn't match.
            remapped_state_dict[parameter_name] = tensor

    # Load into the underlying HF model used by SentenceTransformer
    missing_keys, unexpected_keys = sentence_transformer_model.load_state_dict(
        remapped_state_dict,
        strict=False
    )
    if missing_keys:
        print(f"[Info] Missing keys when loading backbone (expected for dropped heads): {len(missing_keys)}")
    if unexpected_keys:
        print(f"[Info] Unexpected keys ignored: {len(unexpected_keys)}")

    sentence_transformer_model.eval()
    return sentence_transformer_model

_POLY_BERT_CACHE_DIR = "poly_bert_embeddings"
@Memory(location=_POLY_BERT_CACHE_DIR, verbose=0).cache
def _get_poly_bert_embeddings(smiles_df: pd.DataFrame, polybert_embedding_dim_count: int):
    # polybert = SentenceTransformer('/kaggle/input/polybert/polyBERT')

    polybert: SentenceTransformer = load_backbone_into_sentence_transformer(
        base_model_name_or_path="/kaggle/input/polybert/polyBERT",
        finetuned_state_dict_path="/kaggle/input/2-stage-polymer-bert/20250912_233656_poly_6epochs_2thresh_v2.1_rankup/single_split/polymer_bert_rankup.pth",
        device="cuda"
    )
    
    embeddings = polybert.encode(smiles_df['SMILES'].to_list())
    
    embedding_col_names = [f'polyBERT_{index}' for index in range(len(embeddings[0]))]
    features_df = pd.DataFrame(embeddings, columns=embedding_col_names)
    
    # ranked_features = ['polyBERT_360', 'polyBERT_68', 'polyBERT_32', 'polyBERT_207', 'polyBERT_127', 'polyBERT_348', 'polyBERT_189', 'polyBERT_285', 'polyBERT_424', 'polyBERT_415', 'polyBERT_51', 'polyBERT_384', 'polyBERT_210', 'polyBERT_492', 'polyBERT_204', 'polyBERT_410', 'polyBERT_203', 'polyBERT_402', 'polyBERT_50', 'polyBERT_88', 'polyBERT_457', 'polyBERT_584', 'polyBERT_112', 'polyBERT_295', 'polyBERT_544', 'polyBERT_190', 'polyBERT_408', 'polyBERT_338', 'polyBERT_54', 'polyBERT_179', 'polyBERT_246', 'polyBERT_471', 'polyBERT_540', 'polyBERT_280', 'polyBERT_428', 'polyBERT_512', 'polyBERT_82', 'polyBERT_275', 'polyBERT_417', 'polyBERT_154', 'polyBERT_449', 'polyBERT_554', 'polyBERT_74', 'polyBERT_396', 'polyBERT_91', 'polyBERT_208', 'polyBERT_375', 'polyBERT_288', 'polyBERT_201', 'polyBERT_400', 'polyBERT_289', 'polyBERT_134', 'polyBERT_468', 'polyBERT_183', 'polyBERT_228', 'polyBERT_0', 'polyBERT_171', 'polyBERT_436', 'polyBERT_95', 'polyBERT_151', 'polyBERT_297', 'polyBERT_36', 'polyBERT_186', 'polyBERT_316', 'polyBERT_81', 'polyBERT_463', 'polyBERT_302', 'polyBERT_227', 'polyBERT_78', 'polyBERT_168', 'polyBERT_145', 'polyBERT_3', 'polyBERT_170', 'polyBERT_423', 'polyBERT_571', 'polyBERT_301', 'polyBERT_176', 'polyBERT_432', 'polyBERT_60', 'polyBERT_552', 'polyBERT_262', 'polyBERT_300', 'polyBERT_93', 'polyBERT_169', 'polyBERT_191', 'polyBERT_160', 'polyBERT_503', 'polyBERT_380', 'polyBERT_451', 'polyBERT_548', 'polyBERT_57', 'polyBERT_570', 'polyBERT_332', 'polyBERT_441', 'polyBERT_99', 'polyBERT_478', 'polyBERT_477', 'polyBERT_394', 'polyBERT_58', 'polyBERT_308', 'polyBERT_118', 'polyBERT_85', 'polyBERT_101', 'polyBERT_84', 'polyBERT_475', 'polyBERT_440', 'polyBERT_299', 'polyBERT_397', 'polyBERT_325', 'polyBERT_327', 'polyBERT_244', 'polyBERT_1', 'polyBERT_100', 'polyBERT_209', 'polyBERT_343', 'polyBERT_109', 'polyBERT_226', 'polyBERT_21', 'polyBERT_370', 'polyBERT_367', 'polyBERT_23', 'polyBERT_193', 'polyBERT_476', 'polyBERT_369', 'polyBERT_556', 'polyBERT_357', 'polyBERT_335', 'polyBERT_511', 'polyBERT_597', 'polyBERT_494', 'polyBERT_309', 'polyBERT_517', 'polyBERT_562', 'polyBERT_376', 'polyBERT_165', 'polyBERT_336', 'polyBERT_293', 'polyBERT_546', 'polyBERT_158', 'polyBERT_42', 'polyBERT_591', 'polyBERT_218', 'polyBERT_215', 'polyBERT_458', 'polyBERT_383', 'polyBERT_216', 'polyBERT_337', 'polyBERT_253', 'polyBERT_153', 'polyBERT_496', 'polyBERT_328', 'polyBERT_22', 'polyBERT_578', 'polyBERT_319', 'polyBERT_56', 'polyBERT_196', 'polyBERT_433', 'polyBERT_257', 'polyBERT_86', 'polyBERT_486', 'polyBERT_281', 'polyBERT_594', 'polyBERT_497', 'polyBERT_7', 'polyBERT_66', 'polyBERT_15', 'polyBERT_286', 'polyBERT_290', 'polyBERT_37', 'polyBERT_557', 'polyBERT_312', 'polyBERT_141', 'polyBERT_506', 'polyBERT_304', 'polyBERT_598', 'polyBERT_40', 'polyBERT_466', 'polyBERT_10', 'polyBERT_320', 'polyBERT_264', 'polyBERT_500', 'polyBERT_72', 'polyBERT_501', 'polyBERT_76', 'polyBERT_270', 'polyBERT_434', 'polyBERT_374', 'polyBERT_16', 'polyBERT_558', 'polyBERT_550', 'polyBERT_454', 'polyBERT_260', 'polyBERT_177', 'polyBERT_138', 'polyBERT_28', 'polyBERT_2', 'polyBERT_461', 'polyBERT_223', 'polyBERT_499', 'polyBERT_229', 'polyBERT_513', 'polyBERT_233', 'polyBERT_89', 'polyBERT_490', 'polyBERT_19', 'polyBERT_219', 'polyBERT_514', 'polyBERT_135', 'polyBERT_132', 'polyBERT_474', 'polyBERT_483', 'polyBERT_70', 'polyBERT_339', 'polyBERT_491', 'polyBERT_41', 'polyBERT_46', 'polyBERT_493', 'polyBERT_504', 'polyBERT_157', 'polyBERT_391', 'polyBERT_373', 'polyBERT_566', 'polyBERT_102', 'polyBERT_5', 'polyBERT_205', 'polyBERT_47', 'polyBERT_314', 'polyBERT_479', 'polyBERT_65', 'polyBERT_156', 'polyBERT_63', 'polyBERT_581', 'polyBERT_545', 'polyBERT_276', 'polyBERT_291', 'polyBERT_379', 'polyBERT_429', 'polyBERT_108', 'polyBERT_582', 'polyBERT_350', 'polyBERT_352', 'polyBERT_361', 'polyBERT_251', 'polyBERT_49', 'polyBERT_324', 'polyBERT_27', 'polyBERT_247', 'polyBERT_509', 'polyBERT_577', 'polyBERT_284', 'polyBERT_182', 'polyBERT_372', 'polyBERT_206', 'polyBERT_508', 'polyBERT_87', 'polyBERT_242', 'polyBERT_147', 'polyBERT_534', 'polyBERT_188', 'polyBERT_237', 'polyBERT_502', 'polyBERT_273', 'polyBERT_403', 'polyBERT_298', 'polyBERT_307', 'polyBERT_368', 'polyBERT_61', 'polyBERT_267', 'polyBERT_541', 'polyBERT_322', 'polyBERT_580', 'polyBERT_527', 'polyBERT_238', 'polyBERT_358', 'polyBERT_192', 'polyBERT_199', 'polyBERT_555', 'polyBERT_498', 'polyBERT_495', 'polyBERT_94', 'polyBERT_202', 'polyBERT_453', 'polyBERT_442', 'polyBERT_75', 'polyBERT_38', 'polyBERT_55', 'polyBERT_488', 'polyBERT_29', 'polyBERT_425', 'polyBERT_124', 'polyBERT_447', 'polyBERT_366', 'polyBERT_259', 'polyBERT_404', 'polyBERT_387', 'polyBERT_146', 'polyBERT_537', 'polyBERT_221', 'polyBERT_536', 'polyBERT_178', 'polyBERT_437', 'polyBERT_572', 'polyBERT_163', 'polyBERT_287', 'polyBERT_392', 'polyBERT_45', 'polyBERT_531', 'polyBERT_507', 'polyBERT_243', 'polyBERT_465', 'polyBERT_567', 'polyBERT_510', 'polyBERT_266', 'polyBERT_444', 'polyBERT_194', 'polyBERT_388', 'polyBERT_330', 'polyBERT_586', 'polyBERT_9', 'polyBERT_6', 'polyBERT_122', 'polyBERT_313', 'polyBERT_589', 'polyBERT_239', 'polyBERT_344', 'polyBERT_549', 'polyBERT_200', 'polyBERT_418', 'polyBERT_574', 'polyBERT_560', 'polyBERT_155', 'polyBERT_416', 'polyBERT_214', 'polyBERT_363', 'polyBERT_575', 'polyBERT_482', 'polyBERT_174', 'polyBERT_356', 'polyBERT_305', 'polyBERT_181', 'polyBERT_347', 'polyBERT_217', 'polyBERT_371', 'polyBERT_184', 'polyBERT_364', 'polyBERT_282', 'polyBERT_354', 'polyBERT_590', 'polyBERT_44', 'polyBERT_161', 'polyBERT_59', 'polyBERT_144', 'polyBERT_345', 'polyBERT_180', 'polyBERT_470', 'polyBERT_129', 'polyBERT_469', 'polyBERT_526', 'polyBERT_148', 'polyBERT_125', 'polyBERT_113', 'polyBERT_248', 'polyBERT_351', 'polyBERT_92', 'polyBERT_24', 'polyBERT_43', 'polyBERT_473', 'polyBERT_52', 'polyBERT_409', 'polyBERT_349', 'polyBERT_220', 'polyBERT_274', 'polyBERT_98', 'polyBERT_103', 'polyBERT_53', 'polyBERT_587', 'polyBERT_487', 'polyBERT_543', 'polyBERT_412', 'polyBERT_67', 'polyBERT_230', 'polyBERT_321', 'polyBERT_283', 'polyBERT_30', 'polyBERT_136', 'polyBERT_104', 'polyBERT_445', 'polyBERT_419', 'polyBERT_73', 'polyBERT_456', 'polyBERT_123', 'polyBERT_64', 'polyBERT_333', 'polyBERT_386', 'polyBERT_265', 'polyBERT_472', 'polyBERT_164', 'polyBERT_481', 'polyBERT_128', 'polyBERT_489', 'polyBERT_198', 'polyBERT_568', 'polyBERT_406', 'polyBERT_547', 'polyBERT_20', 'polyBERT_389', 'polyBERT_271', 'polyBERT_225', 'polyBERT_167', 'polyBERT_35', 'polyBERT_579', 'polyBERT_551', 'polyBERT_278', 'polyBERT_139', 'polyBERT_272', 'polyBERT_166', 'polyBERT_398', 'polyBERT_133', 'polyBERT_515', 'polyBERT_8', 'polyBERT_83', 'polyBERT_80', 'polyBERT_462', 'polyBERT_378', 'polyBERT_317', 'polyBERT_310', 'polyBERT_268', 'polyBERT_599', 'polyBERT_14', 'polyBERT_279', 'polyBERT_79', 'polyBERT_254', 'polyBERT_411', 'polyBERT_39', 'polyBERT_329', 'polyBERT_185', 'polyBERT_519', 'polyBERT_25', 'polyBERT_107', 'polyBERT_521', 'polyBERT_120', 'polyBERT_435', 'polyBERT_152', 'polyBERT_385', 'polyBERT_197', 'polyBERT_71', 'polyBERT_443', 'polyBERT_353', 'polyBERT_480', 'polyBERT_126', 'polyBERT_342', 'polyBERT_121', 'polyBERT_77', 'polyBERT_459', 'polyBERT_18', 'polyBERT_583', 'polyBERT_359', 'polyBERT_110', 'polyBERT_13', 'polyBERT_323', 'polyBERT_69', 'polyBERT_296', 'polyBERT_306', 'polyBERT_595', 'polyBERT_341', 'polyBERT_255', 'polyBERT_563', 'polyBERT_175', 'polyBERT_505', 'polyBERT_533', 'polyBERT_539', 'polyBERT_245', 'polyBERT_522', 'polyBERT_250', 'polyBERT_431', 'polyBERT_414', 'polyBERT_173', 'polyBERT_224', 'polyBERT_381', 'polyBERT_149', 'polyBERT_455', 'polyBERT_114', 'polyBERT_249', 'polyBERT_421', 'polyBERT_426', 'polyBERT_240', 'polyBERT_365', 'polyBERT_172', 'polyBERT_117', 'polyBERT_318', 'polyBERT_485', 'polyBERT_211', 'polyBERT_467', 'polyBERT_355', 'polyBERT_382', 'polyBERT_464', 'polyBERT_256', 'polyBERT_4', 'polyBERT_460', 'polyBERT_520', 'polyBERT_346', 'polyBERT_390', 'polyBERT_393', 'polyBERT_62', 'polyBERT_576', 'polyBERT_232', 'polyBERT_524', 'polyBERT_115', 'polyBERT_131', 'polyBERT_484', 'polyBERT_334', 'polyBERT_377', 'polyBERT_303', 'polyBERT_535', 'polyBERT_31', 'polyBERT_142', 'polyBERT_222', 'polyBERT_407', 'polyBERT_263', 'polyBERT_315', 'polyBERT_159', 'polyBERT_326', 'polyBERT_592', 'polyBERT_111', 'polyBERT_565', 'polyBERT_162', 'polyBERT_529', 'polyBERT_569', 'polyBERT_340', 'polyBERT_518', 'polyBERT_235', 'polyBERT_97', 'polyBERT_446', 'polyBERT_150', 'polyBERT_399', 'polyBERT_422', 'polyBERT_241', 'polyBERT_516', 'polyBERT_187', 'polyBERT_137', 'polyBERT_195', 'polyBERT_561', 'polyBERT_12', 'polyBERT_538', 'polyBERT_116', 'polyBERT_236', 'polyBERT_252', 'polyBERT_525', 'polyBERT_401', 'polyBERT_564', 'polyBERT_34', 'polyBERT_105', 'polyBERT_413', 'polyBERT_11', 'polyBERT_106', 'polyBERT_234', 'polyBERT_530', 'polyBERT_277', 'polyBERT_405', 'polyBERT_450', 'polyBERT_438', 'polyBERT_213', 'polyBERT_588', 'polyBERT_420', 'polyBERT_212', 'polyBERT_90', 'polyBERT_439', 'polyBERT_528', 'polyBERT_596', 'polyBERT_292', 'polyBERT_258', 'polyBERT_261', 'polyBERT_430', 'polyBERT_311', 'polyBERT_17', 'polyBERT_26', 'polyBERT_96', 'polyBERT_143', 'polyBERT_119', 'polyBERT_130', 'polyBERT_532', 'polyBERT_395', 'polyBERT_542', 'polyBERT_559', 'polyBERT_33', 'polyBERT_593', 'polyBERT_427', 'polyBERT_294', 'polyBERT_48', 'polyBERT_452', 'polyBERT_573', 'polyBERT_585', 'polyBERT_523', 'polyBERT_362', 'polyBERT_553', 'polyBERT_140', 'polyBERT_269', 'polyBERT_448', 'polyBERT_331', 'polyBERT_231']
    ranked_features = ['polyBERT_169', 'polyBERT_308', 'polyBERT_361', 'polyBERT_246', 'polyBERT_56', 'polyBERT_177', 'polyBERT_10', 'polyBERT_434', 'polyBERT_534', 'polyBERT_294', 'polyBERT_93', 'polyBERT_295', 'polyBERT_14', 'polyBERT_52', 'polyBERT_389', 'polyBERT_597', 'polyBERT_406', 'polyBERT_38', 'polyBERT_166', 'polyBERT_544', 'polyBERT_342', 'polyBERT_375', 'polyBERT_32', 'polyBERT_594', 'polyBERT_459', 'polyBERT_218', 'polyBERT_510', 'polyBERT_134', 'polyBERT_242', 'polyBERT_62', 'polyBERT_396', 'polyBERT_102', 'polyBERT_57', 'polyBERT_379', 'polyBERT_157', 'polyBERT_297', 'polyBERT_37', 'polyBERT_27', 'polyBERT_349', 'polyBERT_572', 'polyBERT_141', 'polyBERT_207', 'polyBERT_193', 'polyBERT_309', 'polyBERT_74', 'polyBERT_384', 'polyBERT_554', 'polyBERT_408', 'polyBERT_1', 'polyBERT_525', 'polyBERT_435', 'polyBERT_39', 'polyBERT_370', 'polyBERT_479', 'polyBERT_289', 'polyBERT_383', 'polyBERT_138', 'polyBERT_108', 'polyBERT_142', 'polyBERT_335', 'polyBERT_280', 'polyBERT_198', 'polyBERT_489', 'polyBERT_36', 'polyBERT_549', 'polyBERT_583', 'polyBERT_70', 'polyBERT_352', 'polyBERT_300', 'polyBERT_186', 'polyBERT_423', 'polyBERT_550', 'polyBERT_576', 'polyBERT_372', 'polyBERT_54', 'polyBERT_500', 'polyBERT_590', 'polyBERT_490', 'polyBERT_299', 'polyBERT_587', 'polyBERT_279', 'polyBERT_400', 'polyBERT_471', 'polyBERT_89', 'polyBERT_144', 'polyBERT_348', 'polyBERT_112', 'polyBERT_191', 'polyBERT_578', 'polyBERT_211', 'polyBERT_360', 'polyBERT_514', 'polyBERT_77', 'polyBERT_557', 'polyBERT_382', 'polyBERT_85', 'polyBERT_537', 'polyBERT_437', 'polyBERT_386', 'polyBERT_481', 'polyBERT_555', 'polyBERT_306', 'polyBERT_367', 'polyBERT_577', 'polyBERT_82', 'polyBERT_333', 'polyBERT_478', 'polyBERT_410', 'polyBERT_5', 'polyBERT_446', 'polyBERT_458', 'polyBERT_363', 'polyBERT_124', 'polyBERT_283', 'polyBERT_265', 'polyBERT_165', 'polyBERT_357', 'polyBERT_8', 'polyBERT_447', 'polyBERT_436', 'polyBERT_99', 'polyBERT_499', 'polyBERT_339', 'polyBERT_507', 'polyBERT_12', 'polyBERT_369', 'polyBERT_531', 'polyBERT_78', 'polyBERT_545', 'polyBERT_491', 'polyBERT_201', 'polyBERT_535', 'polyBERT_338', 'polyBERT_120', 'polyBERT_511', 'polyBERT_92', 'polyBERT_237', 'polyBERT_401', 'polyBERT_494', 'polyBERT_546', 'polyBERT_508', 'polyBERT_50', 'polyBERT_84', 'polyBERT_270', 'polyBERT_456', 'polyBERT_189', 'polyBERT_548', 'polyBERT_209', 'polyBERT_581', 'polyBERT_76', 'polyBERT_187', 'polyBERT_170', 'polyBERT_168', 'polyBERT_536', 'polyBERT_203', 'polyBERT_290', 'polyBERT_51', 'polyBERT_276', 'polyBERT_417', 'polyBERT_293', 'polyBERT_517', 'polyBERT_519', 'polyBERT_515', 'polyBERT_183', 'polyBERT_438', 'polyBERT_527', 'polyBERT_298', 'polyBERT_591', 'polyBERT_588', 'polyBERT_398', 'polyBERT_154', 'polyBERT_573', 'polyBERT_516', 'polyBERT_513', 'polyBERT_161', 'polyBERT_278', 'polyBERT_378', 'polyBERT_58', 'polyBERT_466', 'polyBERT_196', 'polyBERT_118', 'polyBERT_560', 'polyBERT_135', 'polyBERT_87', 'polyBERT_133', 'polyBERT_132', 'polyBERT_445', 'polyBERT_197', 'polyBERT_30', 'polyBERT_9', 'polyBERT_172', 'polyBERT_556', 'polyBERT_136', 'polyBERT_521', 'polyBERT_208', 'polyBERT_286', 'polyBERT_488', 'polyBERT_66', 'polyBERT_453', 'polyBERT_558', 'polyBERT_319', 'polyBERT_113', 'polyBERT_397', 'polyBERT_592', 'polyBERT_216', 'polyBERT_244', 'polyBERT_199', 'polyBERT_256', 'polyBERT_288', 'polyBERT_81', 'polyBERT_540', 'polyBERT_41', 'polyBERT_359', 'polyBERT_388', 'polyBERT_310', 'polyBERT_412', 'polyBERT_495', 'polyBERT_409', 'polyBERT_337', 'polyBERT_415', 'polyBERT_473', 'polyBERT_547', 'polyBERT_180', 'polyBERT_114', 'polyBERT_307', 'polyBERT_347', 'polyBERT_247', 'polyBERT_429', 'polyBERT_267', 'polyBERT_392', 'polyBERT_156', 'polyBERT_97', 'polyBERT_407', 'polyBERT_59', 'polyBERT_190', 'polyBERT_264', 'polyBERT_482', 'polyBERT_7', 'polyBERT_579', 'polyBERT_151', 'polyBERT_0', 'polyBERT_230', 'polyBERT_121', 'polyBERT_150', 'polyBERT_3', 'polyBERT_185', 'polyBERT_304', 'polyBERT_158', 'polyBERT_380', 'polyBERT_559', 'polyBERT_416', 'polyBERT_301', 'polyBERT_314', 'polyBERT_275', 'polyBERT_137', 'polyBERT_11', 'polyBERT_483', 'polyBERT_526', 'polyBERT_475', 'polyBERT_440', 'polyBERT_493', 'polyBERT_552', 'polyBERT_351', 'polyBERT_520', 'polyBERT_29', 'polyBERT_60', 'polyBERT_457', 'polyBERT_316', 'polyBERT_75', 'polyBERT_123', 'polyBERT_455', 'polyBERT_470', 'polyBERT_140', 'polyBERT_472', 'polyBERT_599', 'polyBERT_15', 'polyBERT_110', 'polyBERT_582', 'polyBERT_551', 'polyBERT_243', 'polyBERT_228', 'polyBERT_322', 'polyBERT_506', 'polyBERT_404', 'polyBERT_354', 'polyBERT_311', 'polyBERT_331', 'polyBERT_233', 'polyBERT_336', 'polyBERT_217', 'polyBERT_441', 'polyBERT_171', 'polyBERT_19', 'polyBERT_24', 'polyBERT_425', 'polyBERT_376', 'polyBERT_393', 'polyBERT_164', 'polyBERT_566', 'polyBERT_444', 'polyBERT_68', 'polyBERT_250', 'polyBERT_390', 'polyBERT_353', 'polyBERT_542', 'polyBERT_240', 'polyBERT_104', 'polyBERT_223', 'polyBERT_562', 'polyBERT_152', 'polyBERT_426', 'polyBERT_381', 'polyBERT_175', 'polyBERT_45', 'polyBERT_512', 'polyBERT_528', 'polyBERT_127', 'polyBERT_561', 'polyBERT_318', 'polyBERT_23', 'polyBERT_541', 'polyBERT_259', 'polyBERT_63', 'polyBERT_595', 'polyBERT_443', 'polyBERT_277', 'polyBERT_522', 'polyBERT_464', 'polyBERT_115', 'polyBERT_188', 'polyBERT_148', 'polyBERT_538', 'polyBERT_480', 'polyBERT_153', 'polyBERT_461', 'polyBERT_503', 'polyBERT_179', 'polyBERT_586', 'polyBERT_553', 'polyBERT_269', 'polyBERT_257', 'polyBERT_21', 'polyBERT_262', 'polyBERT_368', 'polyBERT_34', 'polyBERT_44', 'polyBERT_106', 'polyBERT_391', 'polyBERT_43', 'polyBERT_424', 'polyBERT_206', 'polyBERT_61', 'polyBERT_529', 'polyBERT_271', 'polyBERT_65', 'polyBERT_28', 'polyBERT_448', 'polyBERT_449', 'polyBERT_111', 'polyBERT_394', 'polyBERT_451', 'polyBERT_46', 'polyBERT_329', 'polyBERT_79', 'polyBERT_80', 'polyBERT_296', 'polyBERT_96', 'polyBERT_73', 'polyBERT_303', 'polyBERT_162', 'polyBERT_450', 'polyBERT_433', 'polyBERT_6', 'polyBERT_284', 'polyBERT_224', 'polyBERT_427', 'polyBERT_419', 'polyBERT_69', 'polyBERT_98', 'polyBERT_465', 'polyBERT_496', 'polyBERT_492', 'polyBERT_248', 'polyBERT_395', 'polyBERT_287', 'polyBERT_219', 'polyBERT_454', 'polyBERT_502', 'polyBERT_413', 'polyBERT_235', 'polyBERT_580', 'polyBERT_202', 'polyBERT_505', 'polyBERT_484', 'polyBERT_252', 'polyBERT_231', 'polyBERT_160', 'polyBERT_477', 'polyBERT_147', 'polyBERT_315', 'polyBERT_260', 'polyBERT_373', 'polyBERT_485', 'polyBERT_374', 'polyBERT_26', 'polyBERT_131', 'polyBERT_371', 'polyBERT_268', 'polyBERT_249', 'polyBERT_255', 'polyBERT_574', 'polyBERT_305', 'polyBERT_101', 'polyBERT_272', 'polyBERT_163', 'polyBERT_320', 'polyBERT_420', 'polyBERT_428', 'polyBERT_463', 'polyBERT_181', 'polyBERT_364', 'polyBERT_332', 'polyBERT_469', 'polyBERT_4', 'polyBERT_501', 'polyBERT_130', 'polyBERT_421', 'polyBERT_377', 'polyBERT_241', 'polyBERT_476', 'polyBERT_596', 'polyBERT_215', 'polyBERT_90', 'polyBERT_254', 'polyBERT_100', 'polyBERT_328', 'polyBERT_414', 'polyBERT_17', 'polyBERT_468', 'polyBERT_355', 'polyBERT_126', 'polyBERT_530', 'polyBERT_86', 'polyBERT_146', 'polyBERT_385', 'polyBERT_589', 'polyBERT_40', 'polyBERT_200', 'polyBERT_486', 'polyBERT_487', 'polyBERT_266', 'polyBERT_107', 'polyBERT_568', 'polyBERT_344', 'polyBERT_285', 'polyBERT_105', 'polyBERT_35', 'polyBERT_226', 'polyBERT_575', 'polyBERT_504', 'polyBERT_462', 'polyBERT_204', 'polyBERT_251', 'polyBERT_323', 'polyBERT_340', 'polyBERT_176', 'polyBERT_598', 'polyBERT_274', 'polyBERT_64', 'polyBERT_291', 'polyBERT_53', 'polyBERT_143', 'polyBERT_302', 'polyBERT_42', 'polyBERT_346', 'polyBERT_593', 'polyBERT_563', 'polyBERT_149', 'polyBERT_225', 'polyBERT_543', 'polyBERT_16', 'polyBERT_345', 'polyBERT_33', 'polyBERT_103', 'polyBERT_192', 'polyBERT_565', 'polyBERT_178', 'polyBERT_245', 'polyBERT_72', 'polyBERT_432', 'polyBERT_365', 'polyBERT_20', 'polyBERT_326', 'polyBERT_411', 'polyBERT_184', 'polyBERT_67', 'polyBERT_174', 'polyBERT_460', 'polyBERT_239', 'polyBERT_116', 'polyBERT_48', 'polyBERT_167', 'polyBERT_564', 'polyBERT_584', 'polyBERT_210', 'polyBERT_117', 'polyBERT_431', 'polyBERT_263', 'polyBERT_418', 'polyBERT_467', 'polyBERT_47', 'polyBERT_273', 'polyBERT_281', 'polyBERT_570', 'polyBERT_524', 'polyBERT_194', 'polyBERT_122', 'polyBERT_220', 'polyBERT_313', 'polyBERT_571', 'polyBERT_139', 'polyBERT_533', 'polyBERT_253', 'polyBERT_405', 'polyBERT_330', 'polyBERT_321', 'polyBERT_94', 'polyBERT_430', 'polyBERT_49', 'polyBERT_327', 'polyBERT_509', 'polyBERT_261', 'polyBERT_523', 'polyBERT_212', 'polyBERT_532', 'polyBERT_155', 'polyBERT_282', 'polyBERT_119', 'polyBERT_442', 'polyBERT_205', 'polyBERT_343', 'polyBERT_238', 'polyBERT_366', 'polyBERT_88', 'polyBERT_221', 'polyBERT_312', 'polyBERT_229', 'polyBERT_439', 'polyBERT_91', 'polyBERT_317', 'polyBERT_129', 'polyBERT_399', 'polyBERT_71', 'polyBERT_325', 'polyBERT_83', 'polyBERT_324', 'polyBERT_227', 'polyBERT_350', 'polyBERT_18', 'polyBERT_95', 'polyBERT_213', 'polyBERT_125', 'polyBERT_128', 'polyBERT_145', 'polyBERT_234', 'polyBERT_195', 'polyBERT_258', 'polyBERT_402', 'polyBERT_539', 'polyBERT_474', 'polyBERT_292', 'polyBERT_109', 'polyBERT_569', 'polyBERT_358', 'polyBERT_341', 'polyBERT_13', 'polyBERT_567', 'polyBERT_22', 'polyBERT_55', 'polyBERT_214', 'polyBERT_497', 'polyBERT_403', 'polyBERT_31', 'polyBERT_362', 'polyBERT_422', 'polyBERT_182', 'polyBERT_334', 'polyBERT_498', 'polyBERT_2', 'polyBERT_232', 'polyBERT_25', 'polyBERT_159', 'polyBERT_585', 'polyBERT_173', 'polyBERT_356', 'polyBERT_452', 'polyBERT_236', 'polyBERT_387', 'polyBERT_518', 'polyBERT_222']
    features_df = features_df[ranked_features[:polybert_embedding_dim_count]]

    return features_df

def get_features_dataframe(
        smiles_df: pd.DataFrame, 
        morgan_fingerprint_dim: int,
        atom_pair_fingerprint_dim: int,
        torsion_dim: int,
        use_maccs_keys: bool,
        use_graph_features: bool,
        backbone_sidechain_detail_level: int,
        use_extra_backbone_sidechain_features: bool,
        models_directory_path: str | None,
        predicted_features_detail_level: int,
        gemini_features_detail_level: int,
        polybert_embedding_dim_count: int
        ) -> tuple[pl.DataFrame, pl.DataFrame]:
    # COMPUTE "STANDARD" FEATURES.
    features_df = _get_standard_features_dataframe(
        smiles_df, 
        morgan_fingerprint_dim,
        atom_pair_fingerprint_dim,
        torsion_dim,
        use_maccs_keys,
        use_graph_features,
        backbone_sidechain_detail_level,
        use_extra_backbone_sidechain_features,
        gemini_features_detail_level
    )

    # MAYBE COMPUTE PREDICTED SIMULATION RESULT FEATURES.
    if (models_directory_path is not None) and (predicted_features_detail_level > 0):
        predicted_features_df = _get_predicted_features_dataframe(smiles_df, models_directory_path)

        if predicted_features_detail_level == 1:
            IMPORTANT_FEATURE_NAMES = [
                'xgb_ffv',
                'xgb_density_g_cm3',
                'xgb_homopoly_NPR1',
                'xgb_homopoly_NPR2',
                'xgb_homopoly_Eccentricity',
                'xgb_homopoly_Asphericity',
                'xgb_homopoly_SpherocityIndex',
                'xgb_monomer_NPR2',
                'xgb_monomer_NPR1',
                'xgb_monomer_Asphericity',
                'xgb_lambda1_A2',
                'xgb_monomer_SpherocityIndex',
                'xgb_diffusivity_A2_per_ps',
                'xgb_p10_persistence_length',
                'xgb_homopoly_PBF',
                'xgb_lambda2_A2',
                'xgb_voxel_count_occupied',
                'xgb_homopoly_LabuteASA',
                'xgb_occupied_volume_A3',
                'xgb_box_volume_A3'
            ]
            predicted_features_df = predicted_features_df[IMPORTANT_FEATURE_NAMES]

        features_df = pd.concat([features_df, predicted_features_df], axis=1)

    # MAYBE COMPUTE polyBERT FINGERPRINTS.
    if polybert_embedding_dim_count > 0:
        polybert_embeddings_df = _get_poly_bert_embeddings(smiles_df, polybert_embedding_dim_count)
        features_df = pd.concat([features_df, polybert_embeddings_df], axis=1)

    return features_df

2025-09-15 12:25:05.393518: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757939105.724603      59 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757939105.829296      59 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Inference

In [11]:
import numpy as np

tabular_submission_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv')

for target_name in TARGET_NAMES:
    # LOAD MODEL GROUPS.
    preprocessing_configs = targets_to_preprocessing_configs[target_name]
    selected_features = targets_to_selected_features[target_name]
    model_groups = targets_to_model_groups[target_name]
    all_groups_imputers = targets_to_imputers[target_name]

    # GENERATE PREDICTIONS WITH EACH GROUP.
    model_groups_predictions = []
    # for preprocessing_config, feature_names, model_group, group_imputers in zip(preprocessing_configs, selected_features, model_groups, all_groups_imputers):
    for preprocessing_config, feature_names, model_group in zip(preprocessing_configs, selected_features, model_groups):
        print(preprocessing_config)
        # PREPROCESS DATA.
        preprocessing_config['models_directory_path'] = '/kaggle/input/polymer-feature-prediction-models'
        features_df = get_features_dataframe(test_df, **preprocessing_config)
        if feature_names is not None:
            features_df = features_df[feature_names]

        # GENERATE PREDICTIONS.
        model_group_predictions = []
        for model in model_group:
            predictions = model.predict(features_df)
            model_group_predictions.append(predictions)
        
        # RECORD MEAN PREDICTIONS.
        model_group_predictions = np.mean(model_group_predictions, axis=0)
        model_groups_predictions.append(model_group_predictions)        

    # RECORD OVERALL AVERAGE.
    group_weights = TARGET_NAMES_TO_GROUP_WEIGHTS[target_name]
    final_predictions = np.average(model_groups_predictions, weights=group_weights, axis=0)
    tabular_submission_df[target_name] = final_predictions

tabular_submission_df.head()

{'morgan_fingerprint_dim': 512, 'atom_pair_fingerprint_dim': 0, 'torsion_dim': 1024, 'use_maccs_keys': False, 'use_graph_features': True, 'backbone_sidechain_detail_level': 1, 'use_extra_backbone_sidechain_features': True, 'models_directory_path': 'simulations/models', 'predicted_features_detail_level': 2, 'gemini_features_detail_level': 2, 'polybert_embedding_dim_count': 150}


/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/usr/lib/python3.11/pickle.py:1718: UserWarning: [12:25:23] WARNING: /workspace/src/collective/../data/../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


[Info] Unexpected keys ignored: 4


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.7.2. This might lead to breaking code or i

...

{'morgan_fingerprint_dim': 0, 'atom_pair_fingerprint_dim': 2048, 'torsion_dim': 0, 'use_maccs_keys': True, 'use_graph_features': True, 'backbone_sidechain_detail_level': 2, 'use_extra_backbone_sidechain_features': False, 'models_directory_path': 'simulations/models', 'predicted_features_detail_level': 0, 'gemini_features_detail_level': 2, 'polybert_embedding_dim_count': 150}


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.6.1 when using version 1.7.2. This might lead to breaking code or i

{'morgan_fingerprint_dim': 512, 'atom_pair_fingerprint_dim': 2048, 'torsion_dim': 1024, 'use_maccs_keys': True, 'use_graph_features': False, 'backbone_sidechain_detail_level': 2, 'use_extra_backbone_sidechain_features': False, 'models_directory_path': 'simulations/models', 'predicted_features_detail_level': 0, 'gemini_features_detail_level': 0, 'polybert_embedding_dim_count': 150}


/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreesRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.6.1 when using version 1.7.2. This might lead to breaking 

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,160.082260,0.373001,0.201162,1.194973,22.784338
1,1422188626,160.655441,0.374138,0.242493,1.107526,21.356424
2,2032016830,94.468163,0.351452,0.238235,1.139989,20.777567


# Part 2: BERT

In [12]:
# test_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

In [13]:
test_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')
sample_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv')

In [14]:
import pandas as pd
import numpy as np
import warnings
import random
import joblib
import torch
from torch import nn
from tqdm.auto import tqdm
from transformers import PreTrainedModel, AutoConfig, AutoModel, AutoTokenizer
from transformers.activations import ACT2FN
from rdkit import Chem
import gc

warnings.filterwarnings('ignore')
from rdkit import rdBase
rdBase.DisableLog('rdApp.warning')

TARGET_VARIABLES = ["Tg", "FFV", "Tc", "Density", "Rg"]
N_AUGMENTATIONS = 50
RANDOM_STATE = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# --- Model Definition ---
class ContextPooler(nn.Module):
    def __init__(self, hidden_size, dropout_prob, activation_name):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.activation = ACT2FN[activation_name]

    def forward(self, hidden_states):
        context_token = hidden_states[:, 0] # Extract CLS token (first token)

        context_token = self.dropout(context_token)
        pooled_output = self.dense(context_token)
        pooled_output = self.activation(pooled_output)
        return pooled_output

class BertRegressor(nn.Module):
    def __init__(
            self, 
            pretrained_model_path, 
            context_pooler_kwargs = {'hidden_size': 384, 'dropout_prob': 0.144, 'activation_name': 'gelu'}):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(pretrained_model_path)
        self.pooler = ContextPooler(**context_pooler_kwargs)
        
        # Final classification layer
        pooler_output_dim = context_pooler_kwargs['hidden_size']
        self.output = torch.nn.Linear(pooler_output_dim, 1)

    def forward(
            self,
            input_ids,
            attention_mask=None,
            token_type_ids=None,
            position_ids=None):
        outputs = self.backbone(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
        )

        pooled_output = self.pooler(outputs.last_hidden_state)        
        regression_output = self.output(pooled_output)

        return regression_output
    
def augment_smiles(smiles: str, n_augs: int):
    mol = Chem.MolFromSmiles(smiles);
    if mol is None: return [smiles]
    augmented = {smiles};
    for _ in range(n_augs * 2):
        if len(augmented) >= n_augs: break
        aug_smiles = Chem.MolToSmiles(mol, canonical=False, doRandom=True, isomericSmiles=True); augmented.add(aug_smiles)
        # aug_smiles = Chem.MolToSmiles(mol, canonical=False, doRandom=True, isomericSmiles=True, allBondsExplicit=True, allHsExplicit=True); augmented.add(aug_smiles)
    return list(augmented)


In [15]:
import os

def get_predictions_df(root_finetuned_weights_path, foundation_model_path, hidden_size, fold_count, device):
    tokenizer = AutoTokenizer.from_pretrained(foundation_model_path)
    
    bert_predictions_df = pd.DataFrame({'id': test_df['id']})
    for target in TARGET_VARIABLES:
        print(f"    Generating Test predictions with TTA for {target}...")
        all_models_target_preds = []
        for fold_id in range(fold_count):
            # FIND WEIGHTS.
            grouped_by_fold = os.path.exists(f'{root_finetuned_weights_path}/fold_{fold_id}')
            model_directory_path = f'{root_finetuned_weights_path}/fold_{fold_id}' if grouped_by_fold else root_finetuned_weights_path
            raw_state_dict = torch.load(f'{model_directory_path}/polymer_bert_v2_{target}.pth')
            
            # LOAD MODEL & SCALER.
            model = BertRegressor(
                foundation_model_path,
                context_pooler_kwargs={
                    "hidden_size": hidden_size,
                    "dropout_prob": 0.144,
                    "activation_name": "gelu",
                },
                # backbone_kwargs={'device_map': device}
            )
            clean_state_dict = {
                key.removeprefix("_orig_mod."): tensor
                for key, tensor in raw_state_dict.items()
            }
            model.load_state_dict(clean_state_dict)
            
            model = model.to(device).eval()
            scaler = joblib.load(f'{model_directory_path}/scaler_{target}.pkl')
    
            # INFERENCE.
            target_preds = []
            for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
                augmented_smiles_list = augment_smiles(row['SMILES'], N_AUGMENTATIONS)
                inputs = tokenizer(augmented_smiles_list, return_tensors='pt', truncation=True, padding=True, max_length=512)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                
                with torch.no_grad(): 
                    preds = model(**inputs)
                
                scaled_preds = preds.cpu().numpy(); 
                unscaled_preds = scaler.inverse_transform(scaled_preds).flatten(); 
                final_pred = np.median(unscaled_preds)
                target_preds.append(final_pred)
    
            all_models_target_preds.append(target_preds)
    
            # CLEANUP.
            del model, scaler
            gc.collect()
            torch.cuda.empty_cache()
    
        target_preds = np.mean(all_models_target_preds, axis = 0)
        bert_predictions_df[target] = target_preds
    return bert_predictions_df


def build_weighted_submission_from_model_configs(
    model_configs: list[dict],
    model_weights: Sequence[float],
    device: str
) -> pd.DataFrame:
    """
    Calls get_predictions_df(**config) for each model config and returns a weighted
    ensemble submission dataframe with columns ['id', *TARGET_VARIABLES].

    Assumptions:
      - get_predictions_df is defined elsewhere (your original function).
      - TARGET_VARIABLES is defined elsewhere.
      - All predictions are for the same test_df and share the same set of IDs.

    If ID order differs across model outputs, later dataframes are reindexed to match
    the first dataframe's ID order.
    """
    if len(model_configs) == 0:
        raise ValueError("model_configs is empty.")
    if len(model_configs) != len(model_weights):
        raise ValueError("model_configs and model_weights must be the same length.")

    per_config_prediction_dfs = []
    for model_config in model_configs:
        per_config_prediction_df = get_predictions_df(**model_config, device=device)
        per_config_prediction_dfs.append(per_config_prediction_df)

    # Ensure all dataframes have the same ID order; reindex if necessary.
    base_ids = per_config_prediction_dfs[0]["id"].to_numpy()
    for index in range(1, len(per_config_prediction_dfs)):
        df = per_config_prediction_dfs[index]
        if not np.array_equal(base_ids, df["id"].to_numpy()):
            df_aligned = df.set_index("id").reindex(base_ids).reset_index()
            df_aligned.rename(columns={"index": "id"}, inplace=True)
            per_config_prediction_dfs[index] = df_aligned

    submission_df = pd.DataFrame({"id": per_config_prediction_dfs[0]["id"]})
    weights_array = np.asarray(model_weights, dtype=np.float64)

    for target_variable in TARGET_VARIABLES:
        stacked_values = np.column_stack([
            df[target_variable].to_numpy() for df in per_config_prediction_dfs
        ])
        submission_df[target_variable] = np.average(
            stacked_values,
            axis=1,
            weights=weights_array
        )

    return submission_df

# basic_bert_submission_df = build_weighted_submission_from_model_configs(
#     model_configs = [
#         {
#             "root_finetuned_weights_path": "/kaggle/input/chemberta-baseline-models/20250906_191941_modern_et_8",
#             "foundation_model_path": "/kaggle/input/modernbert/pytorch/base/2",
#             "hidden_size": 768,
#             "fold_count": 1,
#         },
#         {
#             "root_finetuned_weights_path": "/kaggle/input/chemberta-baseline-models/20250906_191750_codebert_et_8",
#             "foundation_model_path": "/kaggle/input/codebert-base/transformers/default/1/codebert-base",
#             "hidden_size": 768,
#             "fold_count": 1,
#         },
#     ],
#     model_weights = [1, 1],
#     device='cuda:0'
# )

pseudo_bert_submission_df = build_weighted_submission_from_model_configs(
    model_configs = [
        {
            "root_finetuned_weights_path": "/kaggle/input/2-stage-polymer-bert/20250913_182852_modern_tuned_v2.1",
            "foundation_model_path": "/kaggle/input/modernbert/pytorch/base/2",
            "hidden_size": 768,
            "fold_count": 1,
        },
        {
            "root_finetuned_weights_path": "/kaggle/input/2-stage-polymer-bert/20250913_171719_codebert_tuned_v2.2",
            "foundation_model_path": "/kaggle/input/codebert-base/transformers/default/1/codebert-base",
            "hidden_size": 768,
            "fold_count": 1,
        },
    ],
    model_weights = [1, 1.5],
    device='cuda:1'
)

    Generating Test predictions with TTA for Tg...


  0%|          | 0/3 [00:00<?, ?it/s]

W0915 12:26:48.255000 59 torch/_inductor/utils.py:1137] [1/0] Not enough SMs to use max_autotune_gemm mode


    Generating Test predictions with TTA for FFV...


  0%|          | 0/3 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

    Generating Test predictions with TTA for Density...


  0%|          | 0/3 [00:00<?, ?it/s]

    Generating Test predictions with TTA for Rg...


  0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
# basic_bert_submission_df.head()

In [17]:
pseudo_bert_submission_df.head()

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,160.678696,0.370746,0.193538,1.192280,22.510580
1,1422188626,167.062668,0.375872,0.243910,1.113284,21.441080
2,2032016830,99.971472,0.349545,0.222363,1.130300,20.886523


# Part 3: Uni-Mol 2

In [18]:
def can_embed(smiles_string: str) -> bool:
    # return True
    
    """
    Return True only if RDKit can parse the SMILES *and*
    `AllChem.EmbedMolecule` succeeds (status == 0).

    Any parsing, sanitisation, or embedding error ⇒ False.
    """
    try:
        molecule = Chem.MolFromSmiles(smiles_string)

        if molecule.GetNumAtoms(onlyExplicit=False) > 110:
            return False

        if molecule is None:
            return False
            
        embed_status: int = AllChem.EmbedMolecule(
            molecule,
            maxAttempts=50,
            clearConfs=True,
        )
        return embed_status == 0
    except:
        # traceback.print_exc()
        return False

In [19]:
!mkdir -p /usr/local/lib/python3.11/dist-packages/unimol_tools/weights/modelzoo/84M

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
!cp /kaggle/input/unimol-foundation-models/84M/checkpoint.pt /usr/local/lib/python3.11/dist-packages/unimol_tools/weights/modelzoo/84M

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
from unimol_tools import MolPredict
import polars as pl
import torch
import gc
import pandas as pd

UNIMOL_TARGETS_TO_PATHS = {
    'Rg': '/kaggle/input/uni-mol-2-models/UniMol2_2025_08_17_TabM/Rg',
    'Tc': '/kaggle/input/uni-mol-2-models/UniMol2_2025_09_07_TabM/Tc',
    'Tg': '/kaggle/input/uni-mol-2-models/UniMol2_2025_08_17_TabM/Tg',
    'Density': '/kaggle/input/uni-mol-2-models/UniMol2_2025_08_17_TabM/Density',
}

uni_mol_submission_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv')
uni_mol_submission_df = uni_mol_submission_df.set_index('id')

for target_name, predictor_path in UNIMOL_TARGETS_TO_PATHS.items():
    # PREPROCESS DATA.
    test_df = pl.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')
    subset_df = (
        test_df
        .filter(
            pl.col("SMILES").map_elements(
                can_embed,
                return_dtype=pl.Boolean,
            )
        )
        ['id', 'SMILES']
    )
    preprocessed_data_path = f'{target_name}_SMILES.csv'
    subset_df.write_csv(preprocessed_data_path)

    # LOAD MODEL(s).
    predictor = MolPredict(load_model=predictor_path)

    # INFERENCE.
    def generate_unimol_predictions(seed):
        predictor.config['seed'] = seed
        predictions = predictor.predict(data = preprocessed_data_path)
        new_prediction_series = pd.Series(
            predictions.ravel(),       # → 1‑D
            index=subset_df['id'].to_list(),
            name=target_name,
            dtype="float64",
        )
        return new_prediction_series

    seed_0_preds = generate_unimol_predictions(0)
    seed_42_preds = generate_unimol_predictions(42)
    averaged_preds = (seed_0_preds + seed_42_preds) / 2
    # seed_69_preds = generate_unimol_predictions(69) 
    # averaged_preds = (seed_0_preds + seed_42_preds + seed_69_preds) / 3
    uni_mol_submission_df[target_name] = np.nan
    uni_mol_submission_df[target_name].update(averaged_preds)

    # CLEANUP.
    del predictor
    gc.collect()
    torch.cuda.empty_cache()

uni_mol_submission_df = uni_mol_submission_df.reset_index()
uni_mol_submission_df.head()

2025-09-15 12:27:54 | unimol_tools/data/conformer.py | 437 | INFO | Uni-Mol Tools | Start generating conformers...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if pos

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,143.112762,0,0.201854,1.166136,22.595886
1,1422188626,148.456505,0,0.242593,1.100104,20.517580
2,2032016830,111.470726,0,0.225773,1.115130,20.460409


# Part 4: Post-processing

In [22]:
submission_df = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')


TARGET_VARIABLES = ["Tg", "FFV", "Tc", "Density", "Rg"]

for target_name in TARGET_VARIABLES:
    # submission_df[target_name] = (1*tabular_submission_df[target_name] + 1*basic_bert_submission_df[target_name] + 3*pseudo_bert_submission_df[target_name])/5
    submission_df[target_name] = (1*tabular_submission_df[target_name] + 2*pseudo_bert_submission_df[target_name])/3

In [23]:
def combine_weighted_columns(
    df_always: pd.DataFrame,
    df_sometimes: pd.DataFrame,
    key_column: str,
    value_column: str,
    weight_always: float,
    weight_sometimes: float
) -> pd.DataFrame:
    """
    Combine two dataframes by taking a weighted average of `value_column`.
    - df_always has values always populated (non-NaN).
    - df_sometimes has values that may be NaN.
    - If df_sometimes[value_column] is NaN, fall back to df_always[value_column].
    - If both are NaN, result is NaN.

    Parameters
    ----------
    df_always : pd.DataFrame
        DataFrame with a guaranteed non-NaN column for `value_column`.
    df_sometimes : pd.DataFrame
        DataFrame where `value_column` may contain NaN values.
    key_column : str
        Column to join on.
    value_column : str
        Column to combine.
    weight_always : float
        Weight for values from df_always.
    weight_sometimes : float
        Weight for values from df_sometimes.

    Returns
    -------
    pd.DataFrame
        Combined DataFrame with columns [key_column, value_column].
    """
    merged = df_always.merge(
        df_sometimes,
        on=key_column,
        suffixes=("_always", "_sometimes")
    )

    col_always = f"{value_column}_always"
    col_sometimes = f"{value_column}_sometimes"

    merged[value_column] = np.where(
        merged[col_sometimes].notna(),
        weight_always * merged[col_always] + weight_sometimes * merged[col_sometimes],
        merged[col_always]
    )

    return merged[[key_column, value_column]]

for target in TARGET_VARIABLES:
    if target == 'FFV':
        continue
    
    merged_df = combine_weighted_columns(
        df_always = submission_df,
        df_sometimes = uni_mol_submission_df,
        key_column = 'id',
        value_column = target,
        # weight_always = (5.0 / 6.0),
        # weight_sometimes = (1.0 / 6.0)
        weight_always = 0.75,
        weight_sometimes = 0.25
    )
    submission_df[target] = merged_df[target]

submission_df.head()

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,1109053969,*Oc1ccc(C=NN=Cc2ccc(Oc3ccc(C(c4ccc(*)cc4)(C(F)(F)F)C(F)(F)F)cc3)cc2)cc1,156.138103,0.371498,0.197523,1.186417,22.600346
1,1422188626,*Oc1ccc(C(C)(C)c2ccc(Oc3ccc(C(=O)c4cccc(C(=O)c5ccc(*)cc5)c4)cc3)cc2)cc1,160.809320,0.375294,0.243226,1.108549,21.189041
2,2032016830,*c1cccc(OCCCCCCCCOc2cccc(N3C(=O)c4ccc(-c5cccc6c5C(=O)N(*)C6=O)cc4C3=O)c2)c1,101.470458,0.350181,0.227183,1.128930,20.752756


In [24]:
# submission_df["Tg"] += (submission_df["Tg"].std() * 0.1)

In [25]:
display(tabular_submission_df)
# display(basic_bert_submission_df)
display(pseudo_bert_submission_df)
display(uni_mol_submission_df)
submission_df.head()

,id,Tg,FFV,Tc,Density,Rg
0,1109053969,160.082260,0.373001,0.201162,1.194973,22.784338
1,1422188626,160.655441,0.374138,0.242493,1.107526,21.356424
2,2032016830,94.468163,0.351452,0.238235,1.139989,20.777567


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,160.678696,0.370746,0.193538,1.192280,22.510580
1,1422188626,167.062668,0.375872,0.243910,1.113284,21.441080
2,2032016830,99.971472,0.349545,0.222363,1.130300,20.886523


,id,Tg,FFV,Tc,Density,Rg
0,1109053969,143.112762,0,0.201854,1.166136,22.595886
1,1422188626,148.456505,0,0.242593,1.100104,20.517580
2,2032016830,111.470726,0,0.225773,1.115130,20.460409


,id,SMILES,Tg,FFV,Tc,Density,Rg
0,1109053969,*Oc1ccc(C=NN=Cc2ccc(Oc3ccc(C(c4ccc(*)cc4)(C(F)(F)F)C(F)(F)F)cc3)cc2)cc1,156.138103,0.371498,0.197523,1.186417,22.600346
1,1422188626,*Oc1ccc(C(C)(C)c2ccc(Oc3ccc(C(=O)c4cccc(C(=O)c5ccc(*)cc5)c4)cc3)cc2)cc1,160.809320,0.375294,0.243226,1.108549,21.189041
2,2032016830,*c1cccc(OCCCCCCCCOc2cccc(N3C(=O)c4ccc(-c5cccc6c5C(=O)N(*)C6=O)cc4C3=O)c2)c1,101.470458,0.350181,0.227183,1.128930,20.752756


In [26]:
# submission_df["Tg"] += (submission_df["Tg"].std() * 0.5644)

baseline_increment_per_row = submission_df["Tg"].std() * 0.5644
row_count = len(submission_df)

tg_rank = submission_df["Tg"].rank(method="average", ascending=True)
normalized_rank = (tg_rank - 1) / (row_count - 1)          # 0 for smallest ... 1 for largest
scale_factor_per_row = 1.5 - normalized_rank               # 1.5 → 0.5 linearly
offset_per_row = baseline_increment_per_row * scale_factor_per_row

display(offset_per_row.head())

total_offset_target = row_count * baseline_increment_per_row
correction_factor = total_offset_target / offset_per_row.sum()
offset_per_row *= correction_factor

submission_df["Tg"] += offset_per_row

submission_df.head()

0    18.621597
1     9.310799
2    27.932396
Name: Tg, dtype: float64

,id,SMILES,Tg,FFV,Tc,Density,Rg
0,1109053969,*Oc1ccc(C=NN=Cc2ccc(Oc3ccc(C(c4ccc(*)cc4)(C(F)(F)F)C(F)(F)F)cc3)cc2)cc1,174.759700,0.371498,0.197523,1.186417,22.600346
1,1422188626,*Oc1ccc(C(C)(C)c2ccc(Oc3ccc(C(=O)c4cccc(C(=O)c5ccc(*)cc5)c4)cc3)cc2)cc1,170.120119,0.375294,0.243226,1.108549,21.189041
2,2032016830,*c1cccc(OCCCCCCCCOc2cccc(N3C(=O)c4ccc(-c5cccc6c5C(=O)N(*)C6=O)cc4C3=O)c2)c1,129.402854,0.350181,0.227183,1.128930,20.752756


In [27]:
submission_df.to_csv('submission.csv', index=False)